# Workbook 16 — SMS Customer Activation Analytics

## Customer Activation Intelligence System™

This workbook builds a customer activation intelligence layer for the Pizza House Grand Reopening SMS campaign.

The goal is to connect campaign delivery data, customer replies, verified engagement signals, and future marketing segmentation into one structured analytics system.

This workbook expands the Pizza House portfolio beyond restaurant revenue analysis by measuring how customer outreach performed after the move to **5050 Stockton Blvd**.

## 16.0 Business Context

Pizza House moved to a new location at **5050 Stockton Blvd** and launched a Grand Reopening SMS campaign using SimpleTexting.

Earlier Pizza House workbooks focused on orders, revenue, demand timing, customer records, and Tableau reporting. Workbook 16 focuses on customer activation.

The campaign data includes structured delivery reports, campaign summary screenshots, and unstructured reply screenshots. Because SimpleTexting does not provide a direct reply export, this workbook treats reply screenshots as a source of customer intelligence.

The final objective is to build a reusable marketing database that identifies delivered messages, verified customer responses, opt-outs, questions, invalid numbers, duplicate contacts, and future marketing-ready customers.

## 16.1 Customer Activation Framework

Workbook 16 follows a customer activation funnel:

```text
Customer List
    ↓
SMS Delivery
    ↓
Customer Replies
    ↓
Verified Customer Responses
    ↓
Customer Master Database
    ↓
Future Marketing Audience
```

The key business metric is **Verified Customer Responses**, not only exact "YES" replies.

Because coupon responses were delayed, some customers replied multiple times or used signals such as thumbs up, positive emojis, and HELP messages while waiting for the coupon. These are treated as verified engagement when the customer intent is clearly positive.

## 16.2 Setup — Imports and File Paths

This section defines the project folders used throughout Workbook 16.

The notebook follows the existing Pizza House repository structure and keeps source files, cleaned datasets, and final exports separated.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CLEANED_DIR = DATA_DIR / "cleaned"
EXPORT_DIR = DATA_DIR / "exports"

SMS_DIR = RAW_DIR / "sms"
SC_DIR = SMS_DIR / "campaign_summaries"
DR_DIR = SMS_DIR / "delivery_reports"
RES_DIR = SMS_DIR / "replies"
IMPORTS_DIR = SMS_DIR / "imports"

folders = [
    RAW_DIR,
    CLEANED_DIR,
    EXPORT_DIR,
    SMS_DIR,
    SC_DIR,
    DR_DIR,
    RES_DIR,
    IMPORTS_DIR,
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("WB16 folders ready")
print("Campaign summaries:", SC_DIR)
print("Delivery reports:", DR_DIR)
print("Reply screenshots:", RES_DIR)
print("Imports:", IMPORTS_DIR)

## 16.3 Source Data Inventory

This section confirms the files available for Workbook 16.

The source data is grouped into four categories:

- Campaign summary screenshots
- Delivery report CSV files
- Reply screenshots
- Future import or suppression files

This inventory step creates a quick audit trail before analysis begins.

In [ ]:
sc_files = sorted(SC_DIR.glob("*"))
dr_files = sorted(DR_DIR.glob("*.csv"))
res_files = sorted(RES_DIR.glob("*"))
import_files = sorted(IMPORTS_DIR.glob("*"))

print("Campaign summary files:", len(sc_files))
for file in sc_files:
    print(" -", file.name)

print("\nDelivery report files:", len(dr_files))
for file in dr_files:
    print(" -", file.name)

print("\nReply screenshot files:", len(res_files))
for file in res_files:
    print(" -", file.name)

print("\nImport files:", len(import_files))
for file in import_files:
    print(" -", file.name)

## 16.4 Campaign Timeline

The campaign was launched in multiple waves because SimpleTexting daily sending limits required the customer list to be split into smaller batches.

The final campaign structure included:

```text
D1_A
D1_B
D2
D3
D3_2
D4
D5
D6
```

The timeline is documented manually because it explains how the campaign evolved and why multiple campaign batches exist.

In [ ]:
campaign_timeline = pd.DataFrame([
    {
        "campaign": "D1_A",
        "campaign_group": "Initial Test",
        "description": "First launch batch after Pizza House reopening message was prepared.",
        "notes": "Small batch used to begin campaign delivery."
    },
    {
        "campaign": "D1_B",
        "campaign_group": "Initial Test",
        "description": "Second launch batch sent after D1_A.",
        "notes": "Continued initial campaign rollout."
    },
    {
        "campaign": "D2",
        "campaign_group": "Scale Wave",
        "description": "Larger campaign wave after initial batches.",
        "notes": "Daily limits and platform behavior required campaign management."
    },
    {
        "campaign": "D3",
        "campaign_group": "Restart Wave",
        "description": "Campaign wave affected by campaign restart / overlap behavior.",
        "notes": "Tracked separately to preserve accurate source attribution."
    },
    {
        "campaign": "D3_2",
        "campaign_group": "System Constraint Wave",
        "description": "Additional D3-related campaign batch created because of platform constraints.",
        "notes": "Kept as its own campaign to avoid mixing source files."
    },
    {
        "campaign": "D4",
        "campaign_group": "Weekend Wave",
        "description": "Follow-up campaign batch after earlier waves.",
        "notes": "Part of continued customer activation rollout."
    },
    {
        "campaign": "D5",
        "campaign_group": "Final Wave",
        "description": "Final large customer activation wave.",
        "notes": "Sent after campaign structure was stabilized."
    },
    {
        "campaign": "D6",
        "campaign_group": "Final Wave",
        "description": "Final remaining customer activation wave.",
        "notes": "Completed remaining customer outreach."
    },
])

campaign_timeline.to_csv(CLEANED_DIR / "campaign_timeline.csv", index=False)

print("Exported:", CLEANED_DIR / "campaign_timeline.csv")

campaign_timeline

## 16.5 Campaign Summary Table

SimpleTexting campaign summary screenshots provide campaign-level information that is not available through delivery report exports.

This section creates a structured campaign summary table that can be updated from screenshots.

The table is intentionally manual because the screenshot metrics must be verified before they are used in executive reporting.

In [ ]:
campaign_summary = pd.DataFrame([
    {
        "campaign": "D1_A",
        "campaign_group": "Initial Test",
        "contacts": 498,
        "send_date": "2026-06-08",
        "send_time": "15:30",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
    {
        "campaign": "D1_B",
        "campaign_group": "Initial Test",
        "contacts": 493,
        "send_date": "2026-06-08",
        "send_time": "15:45",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
    {
        "campaign": "D2",
        "campaign_group": "Scale Wave",
        "contacts": 2468,
        "send_date": "2026-06-09",
        "send_time": "various",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
    {
        "campaign": "D3",
        "campaign_group": "Restart Wave",
        "contacts": 996,
        "send_date": "2026-06-12",
        "send_time": "15:30",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
    {
        "campaign": "D3_2",
        "campaign_group": "System Constraint Wave",
        "contacts": np.nan,
        "send_date": "2026-06-10",
        "send_time": "various",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update contact and delivery metrics from screenshots."
    },
    {
        "campaign": "D4",
        "campaign_group": "Weekend Wave",
        "contacts": 1501,
        "send_date": "2026-06-14",
        "send_time": "14:00",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
    {
        "campaign": "D5",
        "campaign_group": "Final Wave",
        "contacts": 1509,
        "send_date": "2026-06-16",
        "send_time": "16:00",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
    {
        "campaign": "D6",
        "campaign_group": "Final Wave",
        "contacts": 1511,
        "send_date": "2026-06-17",
        "send_time": "16:00",
        "total_sent": np.nan,
        "delivered": np.nan,
        "failed": np.nan,
        "success_rate": np.nan,
        "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."
    },
])

campaign_summary["send_date"] = pd.to_datetime(campaign_summary["send_date"])

campaign_summary.to_csv(CLEANED_DIR / "campaign_summary.csv", index=False)
campaign_summary.to_csv(EXPORT_DIR / "campaign_summary.csv", index=False)

print("Exported:", CLEANED_DIR / "campaign_summary.csv")
print("Exported:", EXPORT_DIR / "campaign_summary.csv")

campaign_summary

## 16.6 Delivery Report Pipeline

Delivery reports are the structured SMS source data exported from SimpleTexting.

This section imports each delivery report, standardizes column names, attaches the campaign name from the file name, and appends all reports into one delivery master table.

In [ ]:
delivery_frames = []

for path in sorted(DR_DIR.glob("*_dr.csv")):
    campaign = path.stem.replace("_dr", "").upper()

    temp = pd.read_csv(path)

    temp.columns = (
        temp.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    temp["campaign"] = campaign
    temp["source_file"] = path.name

    delivery_frames.append(temp)

    print(f"Loaded {campaign}: {len(temp):,} rows")

if delivery_frames:
    delivery_master = pd.concat(delivery_frames, ignore_index=True)
else:
    delivery_master = pd.DataFrame()
    print("No delivery report files found.")

print("Delivery master records:", len(delivery_master))

delivery_master.head()

## 16.7 Delivery Data Cleaning

This section standardizes phone numbers and delivery statuses.

The cleaned fields make it possible to summarize delivery performance, identify invalid records, find duplicate contacts, and later merge delivery data with customer replies.

In [ ]:
if not delivery_master.empty:
    phone_candidates = [col for col in delivery_master.columns if "phone" in col]
    phone_col = phone_candidates[0] if phone_candidates else None

    if phone_col:
        delivery_master["phone_clean"] = (
            delivery_master[phone_col]
            .astype(str)
            .str.replace(r"\D", "", regex=True)
        )
    else:
        delivery_master["phone_clean"] = np.nan

    status_candidates = [col for col in delivery_master.columns if "status" in col]
    status_col = status_candidates[0] if status_candidates else None

    if status_col:
        delivery_master["delivery_status_clean"] = (
            delivery_master[status_col]
            .astype(str)
            .str.strip()
            .str.lower()
        )
    else:
        delivery_master["delivery_status_clean"] = np.nan

    delivery_master.to_csv(CLEANED_DIR / "delivery_report_master.csv", index=False)

    print("Exported:", CLEANED_DIR / "delivery_report_master.csv")
    print("Phone column used:", phone_col)
    print("Status column used:", status_col)
else:
    print("Delivery master is empty. Add delivery report CSV files before running this section.")

delivery_master.head()

## 16.8 Delivery Status Summary

This section summarizes delivery status by campaign.

The output supports executive reporting and helps identify campaign batches with failed, invalid, or undelivered numbers.

In [ ]:
if not delivery_master.empty:
    delivery_status_summary = (
        delivery_master
        .groupby(["campaign", "delivery_status_clean"], dropna=False)
        .size()
        .reset_index(name="record_count")
        .sort_values(["campaign", "record_count"], ascending=[True, False])
    )

    delivery_status_summary.to_csv(
        CLEANED_DIR / "delivery_status_summary.csv",
        index=False
    )

    print("Exported:", CLEANED_DIR / "delivery_status_summary.csv")
else:
    delivery_status_summary = pd.DataFrame(
        columns=["campaign", "delivery_status_clean", "record_count"]
    )

    print("Delivery status summary not created because delivery master is empty.")

delivery_status_summary

## 16.9 Reply Screenshot Inventory

SimpleTexting does not provide a structured export of customer replies.

Reply screenshots are therefore treated as raw source files. This inventory creates an audit trail of the screenshots available for OCR-assisted extraction and manual quality review.

In [ ]:
reply_inventory = pd.DataFrame({
    "source_file": [file.name for file in sorted(RES_DIR.glob("*"))],
    "path": [str(file) for file in sorted(RES_DIR.glob("*"))],
})

reply_inventory.to_csv(
    CLEANED_DIR / "reply_screenshot_inventory.csv",
    index=False
)

print("Exported:", CLEANED_DIR / "reply_screenshot_inventory.csv")
print("Reply screenshot files:", len(reply_inventory))

reply_inventory

## 16.10 Verified Customer Response Framework

Workbook 16 measures **Verified Customer Responses** instead of only exact "YES" replies.

This business rule reflects how the campaign actually performed. Because coupon responses were delayed, some customers sent additional confirmations, emoji responses, or HELP messages while waiting for the coupon.

A verified customer response includes positive intent to redeem or engage with the promotion.

Examples include:

```text
YES
Yes
Y
Yep
Yeah
Sí
OK
Okay
👍
Positive emoji responses
HELP related to delayed coupon response
Additional positive follow-up confirmations
```

Multiple positive replies from the same customer should count as one verified customer response unless the conversation clearly shows a different intent.

In [ ]:
verified_response_terms = [
    "yes",
    "y",
    "yep",
    "yeah",
    "si",
    "sí",
    "ok",
    "okay",
    "👍",
    "❤️",
    "😊",
    "😁",
]

question_terms = [
    "address",
    "cross",
    "where",
    "hours",
    "open",
    "location",
    "when",
    "what time",
]

negative_terms = [
    "no",
    "nah",
    "nope",
    "don't care",
    "dont care",
]

wrong_number_terms = [
    "wrong",
    "remove",
    "not me",
]

print("Verified response terms:", len(verified_response_terms))
print("Question terms:", len(question_terms))
print("Negative terms:", len(negative_terms))
print("Wrong number terms:", len(wrong_number_terms))

## 16.11 Reply Classification Function

This function applies the Workbook 16 business rules to customer reply text.

The function is intentionally readable and journal-friendly. It prioritizes business interpretation over overly complex text processing.

In [ ]:
def classify_reply(text):
    if pd.isna(text):
        return "UNKNOWN"

    value = str(text).strip().lower()

    if value == "":
        return "UNKNOWN"

    if "stop" in value:
        return "STOP"

    if any(term in value for term in wrong_number_terms):
        return "WRONG_NUMBER"

    if value == "help" or "help" in value:
        return "VERIFIED_RESPONSE"

    if (
        value in verified_response_terms
        or value.startswith("yes")
        or any(term in value for term in ["👍", "❤️", "😊", "😁"])
    ):
        return "VERIFIED_RESPONSE"

    if any(term in value for term in question_terms):
        return "QUESTION"

    if value in negative_terms or any(term in value for term in negative_terms):
        return "NEGATIVE"

    return "OTHER"


example_replies = [
    "YES",
    "👍",
    "HELP",
    "What's the cross street?",
    "STOP",
    "wrong number",
    "No",
    "Thanks!",
]

reply_examples = pd.DataFrame({
    "reply_text": example_replies,
    "reply_category": [classify_reply(reply) for reply in example_replies],
})

reply_examples

## 16.12 Reply Tracker Schema

The reply tracker is the structured dataset used to store customer responses extracted from reply screenshots.

The workflow is now OCR-assisted instead of fully manual. Screenshot text will be extracted, classified, reviewed, and then stored in this table.

This prevents the project from becoming manual data entry and turns the reply screenshots into an analytics pipeline.

In [ ]:
reply_tracker = pd.DataFrame(columns=[
    "campaign",
    "source_file",
    "contact_display",
    "phone_clean",
    "reply_text",
    "reply_category",
    "coupon_sent",
    "follow_up_needed",
    "verified_response_flag",
    "notes",
])

reply_tracker.to_csv(
    CLEANED_DIR / "reply_tracker.csv",
    index=False
)

print("Exported:", CLEANED_DIR / "reply_tracker.csv")

reply_tracker

## 16.13 OCR-Assisted Reply Extraction Plan

This section documents the planned extraction approach for reply screenshots.

The raw screenshots will be processed into a structured reply tracker using this workflow:

```text
Reply Screenshots
    ↓
OCR Text Extraction
    ↓
Conversation Parsing
    ↓
Reply Classification
    ↓
Human Quality Review
    ↓
reply_tracker.csv
```

The goal is not to manually type every customer response. The goal is to use automation to create a first-pass reply dataset, then manually review only ambiguous or low-confidence records.

In [ ]:
ocr_extraction_plan = pd.DataFrame([
    {
        "step": 1,
        "process": "Load reply screenshots",
        "output": "Screenshot inventory"
    },
    {
        "step": 2,
        "process": "Extract visible text from each screenshot",
        "output": "Raw OCR text"
    },
    {
        "step": 3,
        "process": "Identify contact display values and reply text",
        "output": "Draft reply records"
    },
    {
        "step": 4,
        "process": "Apply Workbook 16 reply classification rules",
        "output": "Draft reply categories"
    },
    {
        "step": 5,
        "process": "Review ambiguous records manually",
        "output": "Clean reply tracker"
    },
])

ocr_extraction_plan

## 16.14 Data Quality Outputs

Delivery reports can also be used to identify invalid numbers and duplicate phone records.

These outputs are operationally valuable because they improve the quality of future marketing campaigns and reduce wasted sends.

In [ ]:
if not delivery_master.empty:
    invalid_keywords = ["invalid", "failed", "undelivered", "error"]

    invalid_numbers = delivery_master[
        delivery_master["delivery_status_clean"]
        .astype(str)
        .str.contains("|".join(invalid_keywords), na=False)
    ].copy()

    duplicate_phones = (
        delivery_master[delivery_master["phone_clean"].notna()]
        .groupby("phone_clean")
        .size()
        .reset_index(name="record_count")
        .query("record_count > 1")
        .sort_values("record_count", ascending=False)
    )

    invalid_numbers.to_csv(CLEANED_DIR / "invalid_numbers.csv", index=False)
    duplicate_phones.to_csv(CLEANED_DIR / "duplicate_phones.csv", index=False)

    print("Exported:", CLEANED_DIR / "invalid_numbers.csv")
    print("Exported:", CLEANED_DIR / "duplicate_phones.csv")
    print("Invalid number records:", len(invalid_numbers))
    print("Duplicate phone records:", len(duplicate_phones))
else:
    invalid_numbers = pd.DataFrame()
    duplicate_phones = pd.DataFrame()

    print("Data quality outputs not created because delivery master is empty.")

## 16.15 Initial Activation Funnel

This section creates the first version of the SMS activation funnel using available delivery data.

Reply and verified response metrics will be added after OCR-assisted reply extraction and quality review are complete.

In [ ]:
if not delivery_master.empty:
    total_delivery_records = len(delivery_master)
    unique_phone_records = delivery_master["phone_clean"].nunique()
    invalid_record_count = len(invalid_numbers)

    activation_funnel = pd.DataFrame([
        {
            "stage": "Delivery Report Records",
            "count": total_delivery_records,
        },
        {
            "stage": "Unique Phone Numbers",
            "count": unique_phone_records,
        },
        {
            "stage": "Invalid / Failed Records",
            "count": invalid_record_count,
        },
    ])
else:
    activation_funnel = pd.DataFrame([
        {
            "stage": "Delivery Report Records",
            "count": 0,
        },
        {
            "stage": "Unique Phone Numbers",
            "count": 0,
        },
        {
            "stage": "Invalid / Failed Records",
            "count": 0,
        },
    ])

activation_funnel.to_csv(
    EXPORT_DIR / "sms_activation_funnel.csv",
    index=False
)

print("Exported:", EXPORT_DIR / "sms_activation_funnel.csv")

activation_funnel

## 16.16 Planned Final Outputs

The final version of Workbook 16 will produce a customer activation intelligence layer with operational and executive outputs.

Planned cleaned outputs:

```text
campaign_timeline.csv
campaign_summary.csv
delivery_report_master.csv
delivery_status_summary.csv
reply_screenshot_inventory.csv
reply_tracker.csv
invalid_numbers.csv
duplicate_phones.csv
customer_master.csv
verified_customer_responses.csv
opt_out_customers.csv
customer_questions.csv
wrong_numbers.csv
future_marketing_database.csv
```

Planned export outputs:

```text
sms_activation_funnel.csv
sms_activation_funnel_final.csv
tableau_sms_customer_activation.csv
```

## 16.17 Next Steps

1. Run the starter workbook to verify folder paths and source file inventory.
2. Confirm delivery report imports and delivery status cleaning.
3. Use OCR-assisted extraction to create a first-pass reply tracker from screenshots.
4. Apply the Verified Customer Response framework.
5. Review ambiguous reply records manually.
6. Merge delivery and reply data into `customer_master.csv`.
7. Export action lists for verified responses, opt-outs, questions, wrong numbers, and future marketing.
8. Build the Tableau-ready SMS Customer Activation dataset.
9. Add executive findings and update the GitHub README.